# 01. LangChain tool call


> 랭체인에서도 tool 사용 가능

In [31]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-4o-mini")

llm.invoke([HumanMessage("하이~")])

AIMessage(content='안녕하세요! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 10, 'total_tokens': 20, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_520fb14a9c', 'id': 'chatcmpl-DuuhzbWQgS7deCEJdLcjFFLdAV7ws', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f02b3-317e-75a0-a027-f61513595fe5-0', usage_metadata={'input_tokens': 10, 'output_tokens': 10, 'total_tokens': 20, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [32]:
from langchain_core.tools import tool

In [33]:
from datetime import datetime
import pytz

@tool # @tool 데코레이터를 사용하여 함수를 도구로 등록 / 이 함수에다가 invoke 하기 위해 필요
def get_current_time(timezone: str, location: str) -> str:
    ### 단순한 주석이 아니라 랭체인에 이 함수의 기능과 입력값, 사용 방법을 알려주는 문서화 문자열
    """ 현재 시각을 반환하는 함수 

    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨
    """
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재시각 {now} ' # 타임존, 지역명, 현재시각을 문자열로 반환
    print(location_and_local_time)
    return location_and_local_time


In [36]:
# 도구를 tools 리스트에 추가하고, tool_dict에도 추가
tools = [get_current_time,]
tool_dict = {"get_current_time": get_current_time,}

In [37]:
# 도구를 모델에 바인딩: 모델에 도구를 바인딩하면, 도구를 사용하여 llm 답변을 생성할 수 있음
llm_with_tools = llm.bind_tools(tools)

In [38]:
from langchain_core.messages import SystemMessage

# (4) 사용자의 질문과 tools 사용하여 llm 답변 생성
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("서울은 지금 몇시야?"),
]

# (5) llm_with_tools를 사용하여 사용자의 질문에 대한 llm 답변 생성
response = llm_with_tools.invoke(messages)
messages.append(response)

In [39]:
messages

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='서울은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_8xks84Ebcy4gUNHEP4JCwVdl', 'function': {'arguments': '{"timezone":"Asia/Seoul","location":"Seoul"}', 'name': 'get_current_time'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 134, 'total_tokens': 157, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_b4b9deb02a', 'id': 'chatcmpl-DuuiAZkyZyzcY4YGC76UQh0mSbTkS', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--019f02b3-609e-7831-863c-5e6e6fddbcee-0', tool_

In [40]:
response.tool_calls

[{'name': 'get_current_time',
  'args': {'timezone': 'Asia/Seoul', 'location': 'Seoul'},
  'id': 'call_8xks84Ebcy4gUNHEP4JCwVdl',
  'type': 'tool_call'}]

In [41]:
tool_call = response.tool_calls[0]
tool_call["name"]

'get_current_time'

In [42]:
tool_dict[tool_call["name"]]

StructuredTool(name='get_current_time', description="현재 시각을 반환하는 함수 \n\n   Args:\n       timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함\n       location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨", args_schema=<class 'langchain_core.utils.pydantic.get_current_time'>, func=<function get_current_time at 0x000002DD9D16B1F0>)

In [43]:
tool_dict[tool_call["name"]].invoke(tool_call)

Asia/Seoul (Seoul) 현재시각 2026-06-26 15:52:25 


ToolMessage(content='Asia/Seoul (Seoul) 현재시각 2026-06-26 15:52:25 ', name='get_current_time', tool_call_id='call_8xks84Ebcy4gUNHEP4JCwVdl')

In [44]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # (7) tool_dict를 사용하여 도구 함수를 선택
    print(tool_call["args"]) # (8) 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # (9) 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

messages

{'timezone': 'Asia/Seoul', 'location': 'Seoul'}
Asia/Seoul (Seoul) 현재시각 2026-06-26 15:52:27 


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='서울은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_8xks84Ebcy4gUNHEP4JCwVdl', 'function': {'arguments': '{"timezone":"Asia/Seoul","location":"Seoul"}', 'name': 'get_current_time'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 134, 'total_tokens': 157, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_b4b9deb02a', 'id': 'chatcmpl-DuuiAZkyZyzcY4YGC76UQh0mSbTkS', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--019f02b3-609e-7831-863c-5e6e6fddbcee-0', tool_

In [45]:
llm_with_tools.invoke(messages)

AIMessage(content='서울은 지금 2026년 6월 26일 15시 52분 27초입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 191, 'total_tokens': 216, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_b4b9deb02a', 'id': 'chatcmpl-DuuiMDAhPCBkGKSkm4oojWNdKXIdl', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f02b3-90fc-7163-8fcf-fee4ac450f6d-0', usage_metadata={'input_tokens': 191, 'output_tokens': 25, 'total_tokens': 216, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [46]:
messages.append(llm_with_tools.invoke(messages))

In [47]:
messages

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='서울은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_8xks84Ebcy4gUNHEP4JCwVdl', 'function': {'arguments': '{"timezone":"Asia/Seoul","location":"Seoul"}', 'name': 'get_current_time'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 134, 'total_tokens': 157, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_b4b9deb02a', 'id': 'chatcmpl-DuuiAZkyZyzcY4YGC76UQh0mSbTkS', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--019f02b3-609e-7831-863c-5e6e6fddbcee-0', tool_

### 주가 예제

In [74]:
import yfinance as yf

@tool
def get_yf_stock_history(ticker: str, period: str) -> str:
    """ 
    주식 종목의 가격 데이터를 조회하는 함수

    Args:
        ticker (str): 주식 종목 코드 (예: AAPL)
        period (str): 주식 데이터 조회 기간 (예: 1d, 1mo, 1y)
    
    """
    stock = yf.Ticker(ticker)
    history = stock.history(period=period)
    history_md = history.to_markdown() 

    return history_md

tools = [get_current_time, get_yf_stock_history]
tool_dict = {"get_current_time": get_current_time, "get_yf_stock_history": get_yf_stock_history}

llm_with_tools = llm.bind_tools(tools)

In [75]:
messages.append(HumanMessage("테슬라는 한달 전에 비해 주가가 올랐나 내렸나?"))

response = llm_with_tools.invoke(messages)
messages.append(response)

BadRequestError: Error code: 400 - {'error': {'message': "Invalid parameter: messages with role 'tool' must be a response to a preceeding message with 'tool_calls'.", 'type': 'invalid_request_error', 'param': 'messages.[5].role', 'code': None}}

In [62]:
response.tool_calls

[{'name': 'get_yf_stock_history',
  'args': {'ticker': 'TSLA', 'period': '1mo'},
  'id': 'call_W9hfRalv5ytchC8eaotk5EY2',
  'type': 'tool_call'}]

In [63]:
message

NameError: name 'message' is not defined

#### 이어서 자연어로 모델이 답변하는 것 추가 실습

In [76]:
# (10) 모델이 요청한 tool을 실행하고, 결과를 messages에 추가
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)

# (11) 도구 실행 결과를 참고해 자연어로 최종 답변 생성
final_response = llm_with_tools.invoke(messages)
messages.append(final_response)

final_response.content

BadRequestError: Error code: 400 - {'error': {'message': "Invalid parameter: messages with role 'tool' must be a response to a preceeding message with 'tool_calls'.", 'type': 'invalid_request_error', 'param': 'messages.[5].role', 'code': None}}

In [59]:
!pip install tabulate

In [71]:
#### 이어서 자연어로 모델이 답변하는 것 추가 실습
for tool_call in response.tool_calls:
    out = tool_dict[tool_call['name']].invoke(tool_call)
    messages.append(out)

llm_with_tools.invoke(messages)

BadRequestError: Error code: 400 - {'error': {'message': "Invalid parameter: messages with role 'tool' must be a response to a preceeding message with 'tool_calls'.", 'type': 'invalid_request_error', 'param': 'messages.[5].role', 'code': None}}

## 실습 1. pdf_summary.py

pdf_summary 하는 함수 tool로 등록하여 사용해보기

+ summarize_text 도 langchain 사용하는 것으로 변경

In [67]:
from langchain_core.messages import HumanMessage
import os 

pdf_path = os.path.join(os.getcwd(),"samples/videoanomaly.pdf")

messages = [
    HumanMessage(f"이 {pdf_path} 문서의 저자는 누구야?"),
]


In [68]:
from pdf_summary import summarize_pdf


ValueError: Function must have a docstring if description not provided.

In [ ]:
tools = [summarize_pdf]
tool_dict = {"summarize_pdf": summarize_pdf}
llm_with_tools = llm.bind_tools(tools)


In [ ]:
messages = [HumanMessage(f"이 {pdf_path} 문서의 저자는 누구야?")]

response = llm_with_tools.invoke(messages)
messages.append(response)
response.tool_calls


In [ ]:
for tool_call in response.tool_calls:
    out = tool_dict[tool_call['name']].invoke(tool_call)
    messages.append(out)


In [ ]:
final_response = llm_with_tools.invoke(messages)
final_response.content
